# Part 1 : Snowflake 基礎

## このノートブックでやること
1. 仮想ウェアハウスの作成と設定
2. 外部ステージからのデータ取り込み
3. 【デモ】Snowflake Marketplace からの外部データ活用

## 使用データ
TastyBytes - フードトラック事業のサンプルデータセット


In [ ]:
-- コンテキスト設定
USE DATABASE tb_101;
USE ROLE accountadmin;


## 1. 仮想ウェアハウスの作成と設定

仮想ウェアハウスは、クエリやデータ処理を実行するための**コンピューティングリソース**です。
Snowflake ではストレージとコンピューティングが分離されているため、ウェアハウスを自由にスケールアップ/ダウンできます。

### 主なパラメータ
| パラメータ | 説明 | デフォルト |
|---|---|---|
| `WAREHOUSE_SIZE` | コンピューティングサイズ（X-Small 〜 X-Large 等） | XSmall |
| `AUTO_SUSPEND` | 非アクティブ後に自動停止するまでの秒数 | 600秒 |
| `AUTO_RESUME` | クエリ送信時に自動で再開するか | TRUE |
| `INITIALLY_SUSPENDED` | 作成直後に停止状態で開始するか | TRUE |


In [ ]:
-- 既存のウェアハウスを確認
SHOW WAREHOUSES;


In [ ]:
-- ウェアハウスを作成
-- INITIALLY_SUSPENDED = true → 作成直後は停止状態
-- AUTO_RESUME = false        → 手動で再開が必要（次のステップで体験）
CREATE OR REPLACE WAREHOUSE my_wh
    WAREHOUSE_TYPE      = 'standard'
    WAREHOUSE_SIZE      = 'xsmall'
    AUTO_SUSPEND        = 60
    INITIALLY_SUSPENDED = true
    AUTO_RESUME         = false;


In [ ]:
-- このワークシートで使用するウェアハウスを指定
USE WAREHOUSE my_wh;


In [ ]:
-- ウェアハウスが停止中のままクエリを実行してみる
-- → エラーが返ってくる（停止中は実行できない）
SELECT * FROM raw_pos.truck_details;


エラーが出ましたね。ウェアハウスが停止中のため、クエリを実行できません。

`ALTER WAREHOUSE ... RESUME` で手動再開、または `AUTO_RESUME = TRUE` に設定すればクエリ送信時に自動で再開されます。


In [ ]:
-- ウェアハウスを手動で再開
ALTER WAREHOUSE my_wh RESUME;


In [ ]:
-- 次回から自動再開するよう設定
ALTER WAREHOUSE my_wh SET AUTO_RESUME = TRUE;


In [ ]:
-- 再開後はクエリが実行できる
SELECT * FROM raw_pos.truck_details;


### スケールアップ / スケールダウン

SQL コマンド 1 行でウェアハウスサイズを即座に変更できます。
重いクエリを実行するときだけスケールアップして、終わったらスケールダウン、という使い方が基本です。

> **ポイント:** サイズを 1 段上げるとコンピューティングリソースが 2 倍になります。XSmall → Large で 8 倍。


In [ ]:
-- スケールアップ: XSmall → XLarge
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'XLarge';


In [ ]:
-- XLarge でトラック別売上集計クエリを実行
SELECT
    o.truck_brand_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.price)               AS total_sales
FROM analytics.orders_v o
GROUP BY o.truck_brand_name
ORDER BY total_sales DESC;


In [ ]:
-- 作業が終わったらスケールダウン
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'XSmall';


---
## 2. 外部ステージからのデータ取り込み

Snowflake の**ステージ**は、外部ストレージ（S3 / Azure Blob / GCS）にあるファイルを参照するオブジェクトです。
`COPY INTO` コマンドを使ってステージからテーブルへデータをロードします。

```
S3 バケット
    ↓  CREATE STAGE（参照先を定義）
ステージ (@menu_stage)
    ↓  COPY INTO（テーブルへ取り込み）
テーブル (menu_staging)
```


In [ ]:
-- データエンジニアロールとウェアハウスに切り替え
USE ROLE tb_data_engineer;
USE WAREHOUSE tb_de_wh;


In [ ]:
-- S3 バケットを参照する外部ステージを作成
CREATE OR REPLACE STAGE raw_pos.menu_stage
    COMMENT     = 'メニューデータ用ステージ'
    URL         = 's3://sfquickstarts/frostbyte_tastybytes/raw_pos/menu/'
    FILE_FORMAT = public.csv_ff;


In [ ]:
-- ステージ内のファイル一覧を確認
LIST @raw_pos.menu_stage;


In [ ]:
-- ロード先テーブルを作成
CREATE OR REPLACE TABLE raw_pos.menu_staging
(
    menu_id                      NUMBER(19,0),
    menu_type_id                 NUMBER(38,0),
    menu_type                    VARCHAR(16777216),
    truck_brand_name             VARCHAR(16777216),
    menu_item_id                 NUMBER(38,0),
    menu_item_name               VARCHAR(16777216),
    item_category                VARCHAR(16777216),
    item_subcategory             VARCHAR(16777216),
    cost_of_goods_usd            NUMBER(38,4),
    sale_price_usd               NUMBER(38,4),
    menu_item_health_metrics_obj VARIANT
);


In [ ]:
-- ステージからテーブルへデータをロード
COPY INTO raw_pos.menu_staging
FROM @raw_pos.menu_stage;


In [ ]:
-- ロード結果を確認
SELECT * FROM raw_pos.menu_staging LIMIT 20;


---
## 3. 【デモ】Snowflake Marketplace からの外部データ活用

> **このセクションは講師デモです。**

Snowflake Marketplace では、外部企業が提供するデータを**ゼロコピー**で即座に利用できます。

### デモ内容
- Marketplace から **Weather Source（気象データ）** を取得
- 注文データと気象データを JOIN して「天気と売上の関係」を分析

### ゼロコピーとは
- データを自分のアカウントにコピーしない
- 提供元のデータを直接参照するため、常に最新データを利用可能
- ストレージコストは発生しない

> **強み:** 契約してすぐ使える、メンテナンス不要、常に最新


---
## クリーンアップ

作成したオブジェクトを削除する場合は以下を実行してください。


In [ ]:
USE ROLE accountadmin;
DROP WAREHOUSE IF EXISTS my_wh;
DROP TABLE     IF EXISTS raw_pos.menu_staging;
